In [1]:
import pandas as pd
import re
import nltk

from nltk import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import LancasterStemmer, WordNetLemmatizer
import spacy
from sklearn.preprocessing import LabelEncoder
from num2words import num2words
import unicodedata
from unicodedata import normalize
#import pycontractions

import inflect
import contractions

import warnings
warnings.filterwarnings('ignore')

/Users/kristina/anaconda3/envs/anlp_env/lib/python3.10/site-packages/thinc/compat.py:36: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  hasattr(torch, "has_mps")
/Users/kristina/anaconda3/envs/anlp_env/lib/python3.10/site-packages/thinc/compat.py:37: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  and torch.has_mps  # type: ignore[attr-defined]


In [2]:
# Load English and Portuguese models
en_nlp = spacy.load("en_core_web_sm")
pt_nlp = spacy.load("pt_core_news_sm")

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /Users/kristina/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/kristina/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/kristina/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Data Loading

In [3]:
df_train = pd.read_csv('data/train_zero_shot.csv')
df_test = pd.read_csv('data/test.csv')

In [4]:
df_train.head()

,DataID,Language,MWE,Setting,Previous,Target,Next,Label
0,train_zero_shot.EN.168.1,EN,double dutch,zero_shot,This inspired others to jump ropes as a leisur...,There are several theories behind the origin o...,The most popular theory states that “Double Du...,0
1,train_zero_shot.EN.168.2,EN,double dutch,zero_shot,In the age of chivalry a man paid for the woma...,"Double Dutch also derives from the same era, D...",There are many phrases that include the word: ...,0
2,train_zero_shot.EN.168.3,EN,double dutch,zero_shot,"To her eternal credit, she kept both India and...",Since 1977 we have had a plethora of Foreign M...,We need to exclude from that list the late Mr ...,0
3,train_zero_shot.EN.168.4,EN,double dutch,zero_shot,While pharmaceutical companies were researchin...,Turns out that these people were speaking doub...,So why aren’t Big Macs sold all over the world...,0
4,train_zero_shot.EN.168.5,EN,double dutch,zero_shot,Coronavirus in Europe * Brexit * Brussels ...,Is Flemish premier talking double Dutch?,Three months before the Belgians take over the...,0


In [5]:
df_test.head()

,ID,Language,MWE,Previous,Target,Next,Label,DataID
0,250,EN,bow tie,"With our new system, users can create the trad...",The Bow Tie Inlay System includes four separat...,"The Starter Kit with Frame, Bit and Bushing is...",1,dev.EN.144.2
1,436,EN,life vest,"If you're shopping for a child or children, yo...","They'll be more comfortable, and ultimately sa...",Here are a few of our top picks of kayaking li...,0,dev.EN.275.3
2,450,EN,pillow slip,"""Hurry up and cut a nose,"" Patty whispered. ""I...","""It seems sort of too bad to spoil a perfectly...","""I'll drop some money in the missionary box,"" ...",1,dev.EN.54.9
3,1161,PT,buraco negro,"Agora, um estudo sugere procurar por este “elo...",Estes buracos negros simplesmente não podem ex...,Os buracos negros de intermediária seriam bem ...,0,dev.PT.306.4
4,1331,PT,caixeiro viajante,Ela escreveu um texto acusando o ator em uma c...,"Anna relatou, como colunista convidada da publ...",Ele me pediu para fazer uma massagem em seus p...,1,dev.PT.310.4


In [6]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4491 entries, 0 to 4490
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   DataID    4491 non-null   object
 1   Language  4491 non-null   object
 2   MWE       4491 non-null   object
 3   Setting   4491 non-null   object
 4   Previous  4491 non-null   object
 5   Target    4491 non-null   object
 6   Next      4489 non-null   object
 7   Label     4491 non-null   int64 
dtypes: int64(1), object(7)
memory usage: 280.8+ KB


In [7]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 739 entries, 0 to 738
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   ID        739 non-null    int64 
 1   Language  739 non-null    object
 2   MWE       739 non-null    object
 3   Previous  739 non-null    object
 4   Target    739 non-null    object
 5   Next      739 non-null    object
 6   Label     739 non-null    int64 
 7   DataID    739 non-null    object
dtypes: int64(2), object(6)
memory usage: 46.3+ KB


In [8]:
# Delete NaN values in df since only 2 rows in 'Next' col are missing
df_train = df_train.dropna()
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4489 entries, 0 to 4490
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   DataID    4489 non-null   object
 1   Language  4489 non-null   object
 2   MWE       4489 non-null   object
 3   Setting   4489 non-null   object
 4   Previous  4489 non-null   object
 5   Target    4489 non-null   object
 6   Next      4489 non-null   object
 7   Label     4489 non-null   int64 
dtypes: int64(1), object(7)
memory usage: 315.6+ KB


# Data Pre-Processing
- https://medium.com/analytics-vidhya/building-a-text-classification-model-using-bilstm-c0548ace26f2
- https://medium.com/womenintechnology/from-raw-text-to-insightful-analysis-nlp-text-preprocessing-explained-03dad2d1a3c6
- https://stackoverflow.com/questions/11331982/how-to-remove-any-url-within-a-string-in-python
- https://stackoverflow.com/questions/74403045/handling-stop-words-that-are-part-of-hyphenated-words-while-preprocessing-text?utm_source=chatgpt.com

Discussion on preprocessing choice:

**Punctuation that affects meaning:**
- quotation marks often indicate idioms or direct speech. It makes sense to maintain them here.
- Same applies for Dashes. Used for defininf idiomatic expressions.
- [TBD] square brackets. Needs to be investigated if it carries meaning in examples.

**Capitalization:**
- some idioms have fixed capitalization.

**Numbers:** [TBD] If numbers are part of the idiom, they should be kept

**Lemmatization** could distort idiomatic expressions. 

**Look into**:
- Pycontractions for english & portuguese (custom vs. automated)
- Named Entity Detection

- NLTK port stopwords: https://www.nltk.org/howto/portuguese_en.html
- MWE tokenizer: https://www.geeksforgeeks.org/python-nltk-nltk-tokenize-mwe/
    - https://www.geeksforgeeks.org/python-nltk-nltk-tokenize-mwe/
- Natural Language Processing for Portuguese: https://github.com/pedrobalage/nlppt
- 

In [9]:
df_train.columns = map(str.lower, df_train.columns) # Apply lowercasing to columns
df_test.columns = map(str.lower, df_test.columns)

In [10]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4489 entries, 0 to 4490
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   dataid    4489 non-null   object
 1   language  4489 non-null   object
 2   mwe       4489 non-null   object
 3   setting   4489 non-null   object
 4   previous  4489 non-null   object
 5   target    4489 non-null   object
 6   next      4489 non-null   object
 7   label     4489 non-null   int64 
dtypes: int64(1), object(7)
memory usage: 315.6+ KB


In [11]:
def clean_text(text):
    """Clean input text by removing non-alphanumerics and lowercasing."""
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.lower()

def apply_cleaning(df, cols):
    """Apply clean_text to specified columns of a DataFrame."""
    for col in cols:
        new_col = f"{col}_cleaned"
        df[new_col] = df[col].apply(clean_text)
    return df


In [12]:
columns_to_clean = ['previous', 'target', 'next']
df_train = apply_cleaning(df_train, columns_to_clean)
df_test = apply_cleaning(df_test, columns_to_clean)

In [13]:
# Load English and Portuguese models
en_nlp = spacy.load("en_core_web_sm")
pt_nlp = spacy.load("pt_core_news_sm")

# Get stopwords for both English and Portuguese from NLTK
english_stopwords = set(stopwords.words('english'))
portuguese_stopwords = set(stopwords.words('portuguese'))

In [14]:
# Function to apply tokenization and stopword removal
def preprocess_text(text, lang):
    # Tokenize and remove stopwords based on language
    if lang == 'EN':
        doc = en_nlp(text)
        stopwords_list = english_stopwords
    elif lang == 'PT':
        doc = pt_nlp(text)
        stopwords_list = portuguese_stopwords
    else:
        return text  # Return original text if the language is neither EN nor PT
    
    # Remove stopwords and return tokenized text
    tokens = [token.text for token in doc if token.text.lower() not in stopwords_list and not token.is_space]
    return ' '.join(tokens)

In [15]:
# Apply the preprocessing to the 'previous_cleaned', 'target_cleaned', 'next_cleaned' columns
def apply_preprocessing(df):
    # Loop through rows and apply preprocessing
    for idx, row in df.iterrows():
        lang = row['language']
        
        # Apply preprocessing according to language
        for col in ['previous_cleaned', 'target_cleaned', 'next_cleaned']:
            df.at[idx, col] = preprocess_text(row[col], lang)
    return df

In [16]:
# Apply preprocessing
df_train_preprocessed = apply_preprocessing(df_train)
df_test_preprocessed = apply_preprocessing(df_test)

In [17]:
df_test_preprocessed.tail()

,id,language,mwe,previous,target,next,label,dataid,previous_cleaned,target_cleaned,next_cleaned
734,99713,EN,bow tie,Create a website for your bow tie business.,"Bow ties are primarily worn by men, but both m...",Your site should appeal to both men and women.,1,dev.EN.144.16,create website bow tie business,bow ties primarily worn men men women buy wome...,site appeal men women
735,99741,EN,chain reaction,Grant Norris of Chain Reaction Cycles said: “I...,There’s so much to get involved with throughou...,The bike events world has been hit hard by Cov...,1,dev.EN.11.17,grant norris chain reaction cycles said fantas...,much get involved throughout weekend incredibl...,bike events world hit hard covid tweedlove tea...
736,99763,PT,alta temporada,"Para isso, os dois decidiram se hospedar no On...","Já em alta temporada, que são entre os meses d...","No trajeto para o hotel, Bárbara mostrou parte...",0,dev.PT.289.12,dois decidiram hospedar one nature hotels and ...,j alta temporada so meses junho outubro diria ...,trajeto hotel brbara mostrou parte savana algu...
737,99780,EN,private eye,His family and friends are in mourning.,RELATED:Famous private eye Jack Palladino dies...,Palladino was attacked outside of his Haight-A...,0,dev.EN.170.5,family friends mourning,relatedfamous private eye jack palladino dies ...,palladino attacked outside haightashbury home ...
738,99875,EN,high life,In an announcement released after the market c...,READ: World High Life’s Love Hemp launches pro...,The company said Andrew Male and Robert Paymen...,1,dev.EN.147.16,announcement released market close wednesday c...,read world high lifes love hemp launches produ...,company said andrew male robert payment two ex...


In [18]:
df_test_preprocessed.tail()

,id,language,mwe,previous,target,next,label,dataid,previous_cleaned,target_cleaned,next_cleaned
734,99713,EN,bow tie,Create a website for your bow tie business.,"Bow ties are primarily worn by men, but both m...",Your site should appeal to both men and women.,1,dev.EN.144.16,create website bow tie business,bow ties primarily worn men men women buy wome...,site appeal men women
735,99741,EN,chain reaction,Grant Norris of Chain Reaction Cycles said: “I...,There’s so much to get involved with throughou...,The bike events world has been hit hard by Cov...,1,dev.EN.11.17,grant norris chain reaction cycles said fantas...,much get involved throughout weekend incredibl...,bike events world hit hard covid tweedlove tea...
736,99763,PT,alta temporada,"Para isso, os dois decidiram se hospedar no On...","Já em alta temporada, que são entre os meses d...","No trajeto para o hotel, Bárbara mostrou parte...",0,dev.PT.289.12,dois decidiram hospedar one nature hotels and ...,j alta temporada so meses junho outubro diria ...,trajeto hotel brbara mostrou parte savana algu...
737,99780,EN,private eye,His family and friends are in mourning.,RELATED:Famous private eye Jack Palladino dies...,Palladino was attacked outside of his Haight-A...,0,dev.EN.170.5,family friends mourning,relatedfamous private eye jack palladino dies ...,palladino attacked outside haightashbury home ...
738,99875,EN,high life,In an announcement released after the market c...,READ: World High Life’s Love Hemp launches pro...,The company said Andrew Male and Robert Paymen...,1,dev.EN.147.16,announcement released market close wednesday c...,read world high lifes love hemp launches produ...,company said andrew male robert payment two ex...


In [19]:
df_train_preprocessed.to_csv('data/train_zero_shot_preprocessed.csv', index=False)
df_test_preprocessed.to_csv('data/test_preprocessed.csv', index=False)

### ++++